# Template Definition

In [7]:
ROOT_FOLDER = "../dataset"

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
prompt_template = PromptTemplate.from_template(
"""
You will be asked by the user to create a plant UMl model from specification text. Do so in the most
clear way possible, avoid class properties and assign molteplicity. 

Do include attributes for classes. For example the class Book would be:

class Book{{ String Title, String Author, Date PublicationDate }}

Use only bi-directional arc for relations and no description. For example a relation between
the class Book and the class Page, if the Book can have from one to many pages and the 
pages could have exactly one book, would be:

Book "1..1" -- "1..*" Page

Adapt the cardinality to each case. Where no specific cardinality is specified, use the default "0..*".
If necessary, feel free to use inheritance.
The plantuml has to be the class diagram. In generating the diagram perform this steps in order 

1. Extract class from text
2. Extract attributes for each class
3. Extract relations form text and look for the inheritance
4. Assign the relation to the corresponding class
5. Add cardinality to the relations

Put everything in this order: first all classes and then all relations. In our example would be:

@startuml

class Book{{ String Title, String Author, Date PublicationDate }}
class Page{{ String Content}}

Book "1..1" -- "1..*" Page

@enduml

Output plantuml without futher text or explaination.

##############

Here is an example of how should be done. 

The specificartion example text is:

Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. 
The broker is registered in the system, so that when a customer calls, based on the contract, 
the help desk can immediately trace who is the customer's first account manager.
After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for,
so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file
for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy,
a preliminary contract/offer on an insurance product is made to the customer either in person or by email.
(Such offers can also be extended to already existing customers.) 
If the customer agrees to the offer, the contract is signed by both parties.
After the signing of the contract, the client enjoys the coverage and is invoiced
(monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation.
Then the company opens one or several claim cases (e.g. in case of an accident,
often material damage & physical damage are handled separately). Once the case file is complete, 
it is sent for assessment by different estimators based on their area of expertise.
According to the reports issued by the estimators, it is decided whether the claim case is approved. 
In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.
For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account. 

The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

The corresponding uml is:

@startuml

class Customer {{
    String Name
    String email
}}
class InsurancePolicy {{
    String Type
    Double MonthlyPrice
    Double YearlyPrice
}}
class Contract {{
    String Status
    String InvoiceFrequency
}}
class Invoice {{}}

class Broker{{
    String Name
}}
class Claim {{
    Double CompensationTotal
    Double calculateCompenstationSum()
}}
class ClaimCase {{
    String Status
    String CompensationDecision
}}
class Report {{}}
class Estimator {{
    String AreaOfExpertise
}}
class CompensationPayment {{
    Date PaymentDate
}}

Customer "1"--"0..*" Contract
Contract "1" -- "0..*" Invoice
InsurancePolicy "1" -- "0..*" Contract
Contract "1" -- "0..*" ClaimCase
Customer "1" -- "0..*" Claim
Claim "1" -- "0..*" ClaimCase
ClaimCase "1" -- "0..*" CompensationPayment
Estimator "0..*" - "0..*" ClaimCase
(ClaimCase,Estimator) .. Report
Customer "0..*" - "0..1" Broker


@enduml

##############

The specification text is:

{text}

##############


Based on the example, the uml output is:
"""
)

In [9]:
import sys
import threading
from time import sleep
try:
    import thread
except ImportError:
    import _thread as thread

def quit_function(fn_name):
    # print to stderr, unbuffered in Python 2.
    print('{0} took too long'.format(fn_name), file=sys.stderr)
    sys.stderr.flush() # Python 3 stderr is likely buffered.
    thread.interrupt_main() # raises KeyboardInterrupt
    
def exit_after(s):
    '''
    use as decorator to exit process if 
    function takes longer than s seconds
    '''
    def outer(fn):
        def inner(*args, **kwargs):
            timer = threading.Timer(s, quit_function, args=[fn.__name__])
            timer.start()
            try:
                result = fn(*args, **kwargs)
            finally:
                timer.cancel()
            return result
        return inner
    return outer

In [10]:
import os
from tqdm import tqdm
from docx import Document

from langchain_core.tracers.context import tracing_v2_enabled

@exit_after(360)
def run_chain(chain, text):
    return chain.invoke({"text": text})

def process_subfolders_with_chain(root_folder_path, chain, type=''):
    """
    Explores subfolders of the root folder (depth 1), processes each subfolder's `text.txt`
    with the provided LangChain chain, and saves the result in a new file in the same folder.

    Args:
        root_folder_path (str): Path to the root folder.
        chain: A LangChain chain instance to process text inputs.
    """
    for subfolder_name in tqdm(os.listdir(root_folder_path)):
        subfolder_path = os.path.join(root_folder_path, subfolder_name)
        
        # Ensure the current item is a subfolder
        if os.path.isdir(subfolder_path):
            text_file_path = os.path.join(subfolder_path, "description.md")

            # Recheck if `text.txt` now exists
            if os.path.isfile(text_file_path):
                with open(text_file_path, "r", encoding="utf-8") as file:
                    text = file.read()

                with tracing_v2_enabled():
                    # Call the LangChain chain with the input dictionary
                    try:
                        result = run_chain(chain, text)
                    except:
                        result = ""

                # Save the result to a new file in the same subfolder
                result_file_path = os.path.join(subfolder_path, f"result_few_{type}.txt")
                with open(result_file_path, "w", encoding="utf-8") as result_file:
                    result_file.write(result)

In [11]:
def delete_result_txt_files(root_folder_path):
    """
    Deletes every .txt file that starts with 'result_' in the subfolders of the root folder (depth 1).

    Args:
        root_folder_path (str): Path to the root folder.
    """
    for subfolder_name in os.listdir(root_folder_path):
        subfolder_path = os.path.join(root_folder_path, subfolder_name)
        
        # Ensure the current item is a subfolder
        if os.path.isdir(subfolder_path):
            for file_name in os.listdir(subfolder_path):
                if file_name.startswith("result_") and file_name.endswith(".txt"):
                    file_path = os.path.join(subfolder_path, file_name)
                    os.remove(file_path)

In [12]:
delete_result_txt_files(ROOT_FOLDER)

# One Shot Open-AI

In [13]:
from dotenv import load_dotenv
assert load_dotenv()

In [14]:
MODEL_OPEN_AI = ["gpt-5-mini","gpt-5","gpt-5-nano", "o3-mini", "gpt-4o-mini", "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo"]

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display_markdown

In [16]:
def open_ai_one(model):
    model_ = ChatOpenAI(model=model, reasoning_effort="low")
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [17]:
def open_ai_make_example(model):
    model = ChatOpenAI(model=model, reasoning_effort="low")
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [18]:
display_markdown(open_ai_make_example(MODEL_OPEN_AI[0]), raw=True)

@startuml

class Customer {
    String Name
    String Email
}
class Broker {
    String Name
}
class InsurancePolicy {
    String Type
    Double MonthlyPrice
    Double YearlyPrice
}
class Contract {
    String Status
    String InvoiceFrequency
}
class Invoice {
    Date InvoiceDate
    Double Amount
}
class Claim {
    Double CompensationTotal
}
class ClaimCase {
    String Status
    String CompensationDecision
}
class Estimator {
    String AreaOfExpertise
}
class Report {
    Date CreatedDate
    Date RetentionUntil
}
class CompensationPayment {
    Date PaymentDate
    Double Amount
}

Customer "1" -- "0..*" Contract
Contract "1" -- "0..*" Invoice
InsurancePolicy "1" -- "0..*" Contract
Contract "1" -- "0..*" ClaimCase
Customer "1" -- "0..*" Claim
Claim "1" -- "0..*" ClaimCase
ClaimCase "1" -- "0..*" CompensationPayment
Estimator "0..*" -- "0..*" ClaimCase
ClaimCase "0..*" -- "0..*" Report
Estimator "0..*" -- "0..*" Report
Customer "0..*" -- "0..1" Broker

@enduml

In [19]:
MODEL_OPEN_AI[0:2]

['gpt-5-mini', 'gpt-5']

In [20]:
for model in MODEL_OPEN_AI[0:2wq2]:
    print(f"Few shot with {model}")
    open_ai_one(model)

Few shot with gpt-5-mini


100%|██████████| 48/48 [19:49<00:00, 24.78s/it]


Few shot with gpt-5


100%|██████████| 48/48 [20:45<00:00, 25.95s/it]


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## One Shot Anthropic

In [21]:
from langchain_anthropic import ChatAnthropic

In [22]:
MODEL_ANTHROPIC = ["claude-sonnet-4-5-20250929","claude-3-7-sonnet-20250219"]

In [23]:
def anthropic_make_example(model):
    model = ChatAnthropic(model=model,temperature=0,
    max_tokens=4096,
    timeout=None,
    max_retries=2,)
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [24]:
def anthropic_one(model):
    model_ = ChatAnthropic(model=model)
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [25]:
display_markdown(anthropic_make_example(MODEL_ANTHROPIC[0]), raw=True)

TypeError: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [ ]:
for model in MODEL_ANTHROPIC:
    print(f"One shot with {model}")
    anthropic_one(model)

One shot with claude-sonnet-4-5-20250929


  0%|          | 0/48 [00:00<?, ?it/s]

100%|██████████| 48/48 [04:45<00:00,  5.94s/it]


One shot with claude-3-7-sonnet-20250219


100%|██████████| 48/48 [02:36<00:00,  3.27s/it]


# One shot Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
MODEL_GEMINI = ["gemini-2.5-flash"]

In [ ]:
def gemini_one(model):
    model_ = ChatGoogleGenerativeAI(model=model)
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [ ]:
def gemini_make_example(model):
    model = ChatGoogleGenerativeAI(model=model)
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system\so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [ ]:
display_markdown(gemini_make_example(MODEL_GEMINI[0]), raw=True)

@startuml

class Customer {}
class InsurancePolicy {}
class Contract {}
class Invoice {}
class BrokerCustomerAssignment {}
class Broker {}
class ClaimCase {}
class Report {}
class Estimator {}
class CompensationPayment {}
class InsuranceCompany {}

Broker "1" -- BrokerCustomerAssignment
BrokerCustomerAssignment "0..1" -- "1" Customer
ClaimCase "1" -- CompensationPayment
ClaimCase "1" -- Report
Contract -- "1" Customer
Contract "1" -- ClaimCase
Contract "1" -- InsurancePolicy
Contract "1" -- Invoice
Customer "1" -- ClaimCase
Estimator "1" -- Report
InsuranceCompany -- "1" Contract
InsuranceCompany "1" -- ClaimCase
InsuranceCompany "1" -- InsurancePolicy

@enduml

In [ ]:
for model in MODEL_GEMINI:
    print(f"One shot with {model}")
    gemini_one(model)


One shot with gemini-2.5-flash


100%|██████████| 48/48 [29:57<00:00, 37.44s/it]


# One Shot Open LLM

In [ ]:
import torch, gc

## Deepseek

In [ ]:
from langchain_deepseek import ChatDeepSeek

In [ ]:
MODEL_DEEPSEEK = ["deepseek-chat"] #, "deepseek-reasoner"]

In [ ]:
def deepseek_make_example(model):
    model = ChatDeepSeek(
            model=model,
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
            # api_key="...",
            # other params...
        )
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [ ]:
def deepseek_one(model):
    model_ = ChatDeepSeek(model=model)
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [ ]:
%%time
display_markdown(deepseek_make_example(MODEL_DEEPSEEK[0]), raw=True)

@startuml

class Customer {}
class InsurancePolicy {}
class Contract {}
class Invoice {}
class Broker {}
class ClaimCase {}
class Report {}
class Estimator {}
class CompensationPayment {}
class BrokerCustomerAssignment {}

Contract "1" -- "1" Customer
Contract "1" -- "0..*" Invoice
Contract "1" -- "1..*" InsurancePolicy
Contract "1" -- "0..*" ClaimCase
ClaimCase "1" -- "0..*" CompensationPayment
ClaimCase "1" -- "0..*" Report
Report "1" -- "1" Estimator
BrokerCustomerAssignment "1" -- "1" Customer
Broker "1" -- "0..*" BrokerCustomerAssignment

@enduml

CPU times: user 169 ms, sys: 49.7 ms, total: 218 ms
Wall time: 7.39 s


In [ ]:
for model in MODEL_DEEPSEEK:
    print(f"One shot with {model}")
    deepseek_one(model)

One shot with deepseek-chat


100%|██████████| 48/48 [05:31<00:00,  6.91s/it]


## Mistral

In [ ]:
from langchain_mistralai import ChatMistralAI

In [ ]:
MODEL_MISTRAL = ["mistral-large-2411", "codestral-2508", "mistral-small-2506"]

In [ ]:
def mistral_one(model):
    model_ = ChatMistralAI(model=model)
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [ ]:
def mistral_make_example(model):
    model = ChatMistralAI(model=model)
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system\so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [ ]:
display_markdown(mistral_make_example(MODEL_MISTRAL[0]), raw=True)

@startuml

class Customer {}
class InsurancePolicy {}
class Contract {}
class Invoice {}
class Broker {}
class ClaimCase {}
class Report {}
class Estimator {}
class CompensationPayment {}

Customer "1" -- "0..*" Contract
Contract "1" -- "0..*" Invoice
Contract "1" -- "0..*" InsurancePolicy
Contract "1" -- "0..*" ClaimCase
Customer "1" -- "0..1" Broker
ClaimCase "1" -- "0..*" CompensationPayment
ClaimCase "1" -- "0..*" Report
Report "0..*" -- "1" Estimator

@enduml

In [ ]:
for model in MODEL_MISTRAL:
    print(f"One shot with {model}")
    mistral_one(model)

One shot with mistral-large-2411


100%|██████████| 48/48 [02:47<00:00,  3.48s/it]


One shot with codestral-2508


100%|██████████| 48/48 [00:39<00:00,  1.22it/s]


One shot with mistral-small-2506


100%|██████████| 48/48 [01:37<00:00,  2.03s/it]


## Ollama

In [ ]:
from langchain_ollama import ChatOllama

In [ ]:
MODEL_OLLAMA = [] #["llama3.2:3b-text-fp16"]

In [ ]:
def ollama_one(model):
    model_ = ChatOllama(
        model=model,
        temperature=0,
        timeout = 3,
    )
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [ ]:
def ollama_make_example(model):
    model = ChatOllama(
        model=model,
        temperature=0,
        timeout = 3,
    )
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res
    

In [ ]:
#display_markdown(ollama_make_example(MODEL_OLLAMA[0]), raw=True)

In [ ]:
for model in MODEL_OLLAMA:
    print(f"Zero shot with {model}")
    ollama_one(model)
    gc.collect()
    torch.mps.empty_cache()

## Huggingface

In [ ]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace, HuggingFaceEndpoint
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
MODEL_HUGGINGFACE = [] #["Qwen/Qwen2.5-3B-Instruct", "microsoft/Phi-3-mini-4k-instruct", "google/gemma-2-27b-it"]

In [ ]:
def huggingface_one(model):
    model_id = model
    
    llm = HuggingFaceEndpoint(
        repo_id=model_id,
        task="text-generation",
        max_new_tokens=2048,
        do_sample=False,
        repetition_penalty=1.03,
        temperature=0.01)

    chat = ChatHuggingFace(llm=llm, verbose=True)
    
    chain = prompt_template | chat | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model_id.replace("/","_"))

In [ ]:
def huggingface_make_example(model):

    model_id = model
    
    

    llm = HuggingFacePipeline.from_model_id(
        model_id=model_id,
        task="text-generation",
        pipeline_kwargs={"temperature": 0.1, "max_new_tokens": 1024}
    )

    chat = ChatHuggingFace(llm=llm, verbose=True)
    
    chain = prompt_template | chat | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
        As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.
        
        In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 
        
        """})
    return res


In [ ]:
#display_markdown(huggingface_make_example(MODEL_HUGGINGFACE[0]), raw=True)

In [ ]:
for model in MODEL_HUGGINGFACE:
    print(f"Zero shot with {model}")
    huggingface_one(model)
    gc.collect()
    torch.mps.empty_cache()

## Mlx-LLM

In [ ]:
from langchain_community.llms import MLXPipeline
from langchain_community.chat_models import ChatMLX

In [ ]:
MODEL_MLX = ["mlx-community/phi-4-8bit",
             "mlx-community/Falcon3-10B-Instruct-6bit", 
             "mlx-community/Qwen3-4B-Instruct-2507-4bit-DWQ-2510",
             "mlx-community/Mistral-7B-Instruct-v0.3-4bit",
             "mlx-community/Llama-3.2-3B-Instruct-4bit",
             "mlx-community/gemma-3-4b-it-5bit",
             #"mlx-community/gemma-2-27b-it-4bit",
            # "mlx-community/Mamba-Codestral-7B-v0.1-8bit",
            #"mlx-community/CodeLlama-13b-Instruct-hf-4bit-MLX"
            ]

In [ ]:
def mlx_one(model):
    llm = MLXPipeline.from_model_id(
        model_id=model,
        pipeline_kwargs={"max_tokens": 15_000, "temp": 0.1},
    )
    chat = ChatMLX(llm=llm)
    chain = prompt_template | chat | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model.replace("/","_"))

In [ ]:
def mlx_make_example(model):
    llm = MLXPipeline.from_model_id(
        model_id=model,
        pipeline_kwargs={"max_tokens": 2000, "temp": 0.7},
    )
    chat = ChatMLX(llm=llm)
    chain = prompt_template | chat | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res


In [ ]:
display_markdown(mlx_make_example(MODEL_MLX[2]), raw=True)

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

@startuml

class Customer {}
class InsurancePolicy {}
class Contract {}
class Invoice {}
class Broker {}
class BrokerCustomerAssignment {}
class ClaimCase {}
class Report {}
class Estimator {}
class CompensationPayment {}

Contract "0..*" -- "1" Customer
Contract "1" -- "0..*" Invoice
Contract "1" -- "0..*" InsurancePolicy
BrokerCustomerAssignment "0..1" -- "1" Customer
Broker "1" -- "0..*" BrokerCustomerAssignment
ClaimCase "1" -- "0..*" CompensationPayment
ClaimCase "1" -- "0..*" Report
Report "0..*" -- "1" Estimator

@enduml

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=224c9a71-794b-4d5b-b811-37be09eb0ad7,id=224c9a71-794b-4d5b-b811-37be09eb0ad7; trace=224c9a71-794b-4d5b-b811-37be09eb0ad7,id=eb73750c-6af9-4668-b884-74bfb4de9c66; trace=224c9a71-794b-4d5b-b811-37be09eb0ad7,id=eb73750c-6af9-4668-b884-74bfb4de9c66; trace=224c9a71-794b-4d5b-b811-37be09eb0ad7,id=c6fe24a8-23d3-4fa1-987c-e5daca2bbbc9; trace=224c9a71-794b-4d5b-b811-37be09eb0ad7,id=c6fe24a8-23d3-4fa1-987c-e5daca2bbbc9; trace=224c9a71-794b-4d5b-b811-37be09eb0ad7,id=224c9a71-794b-4d5b-b811-37be09eb0ad7; trace=bce12c38-5408-4f91-9eb1-ccbbbff483fa,id=bce12c38-5408-4f91-9eb1-ccbbbff483fa; trace=bce12c38-5408-4f91-9eb1-ccbbbff483fa

In [ ]:
for model in MODEL_MLX:
    import gc, torch
    gc.collect()
    torch.mps.empty_cache()
    print(f"Few shot with {model}")
    mlx_one(model)
    

Few shot with mlx-community/phi-4-8bit


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 48/48 [15:43<00:00, 19.65s/it]


Few shot with mlx-community/Falcon3-10B-Instruct-6bit


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

...

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 48/48 [35:07<00:00, 43.92s/it]  


Few shot with mlx-community/gemma-3-4b-it-5bit


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

100%|██████████| 48/48 [03:44<00:00,  4.69s/it]


### Clean Cache

In [ ]:
import gc, torch
gc.collect()
torch.mps.empty_cache()